# ENTSO-E Data Cleaning

### Poland (PL)

**Generation:** Remove `Other Renewable`, `Other`, `Hydro Pumped Storage – Actual Consumption`, and `Wind Offshore`.

**Wind/Solar Forecast:** Remove `Wind Offshore`.

**Day-ahead Prices:** Deduplicate timestamps and keep the first occurrence.

**Generation Unavailability:** Use `generation_units_unavailability` with `Planned maintenance` only. Drop `docstatus`; cast `avail_qty` and `pstn` to numeric; remove records with missing `nominal_power` or blank `plant_type`; remove exact duplicates and `(mrid, revision)` duplicates; remove records with `created_doc_time` more than 3 years before `start` or `created_doc_time >= end`.

**Production Unavailability:** Do not use `production_units_unavailability` due to the low number of records.

### Germany / Luxembourg (DE_LU)

**Generation Unavailability:** Use `generation_units_unavailability` with `Planned maintenance` and `Unplanned outage`. Apply the same cleaning procedure as for PL.

**Production Unavailability:** Do not use `production_units_unavailability` because the data starts only from October 2025.

### France (FR)

**Wind/Solar Forecast:** Remove `Wind Offshore`.

**Generation Unavailability:** Use `generation_units_unavailability` with `Planned maintenance` and `Unplanned outage`. Apply the same cleaning procedure as for PL.

**Production Unavailability:** Use `production_units_unavailability` with `Planned maintenance` only. Apply the same cleaning procedure.

### Lithuania (LT)

**Generation Forecast:** Exclude `generation_forecast` entirely due to insufficient data quality.


In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config import (
    BASE_DIR,
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
    ENTSOE_BIDDING_ZONE_CODES,
)
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.cleaning import (
    clean_pl,
    clean_de_lu,
    clean_fr,
    clean_lt,
    save_data_entsoe,
)

In [2]:
output_dir = DATA_DIR / "clean" / "entsoe"
raw_data_dir = DATA_DIR / "raw" / "entsoe"
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
zone_codes = ENTSOE_BIDDING_ZONE_CODES
logging.basicConfig(level=logging.INFO)

In [3]:
data = {
    bz: {
        file.stem: pd.read_parquet(file)
        for file in (raw_data_dir / bz / date_range).iterdir()
    }
    for bz in ENTSOE_BIDDING_ZONE_CODES
}

In [4]:
data["PL"] = clean_pl(data["PL"])
data["DE_LU"] = clean_de_lu(data["DE_LU"])
data["FR"] = clean_fr(data["FR"])
data["LT"] = clean_lt(data["LT"])

codes = ["PL", "DE_LU", "FR", "LT"]

## 1. Index Integrity – Confirmation

The cleaned datasets no longer contain duplicated index values: the 45 exact duplicates in PL `day_ahead_prices` are gone, and so are the duplicated indices in the unavailability tables. The only repeated timestamps left come from the Summer/Winter time switch. Gaps in the time index are not part of this step and are handled later, during preprocessing.

In [5]:
for code in codes:
    print(f"===== {code} {"=" * 20}")
    for n, d in data[code].items():
        print_index_summary(d, n)

===== PL ====================

--- [load] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [generation] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [load_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [day_ahead_prices] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 6
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 33600
  0 days 01:00:00: 24839

--- [generation_wind_solar_forecast] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 9
Timezone:   Europe/Warsaw

Time steps:
  0 days 00:15:00: 79191
  0 days 01:00:00: 13441

--- [generation_

## 2. Missing Values – Confirmation

No missing values remain in PL, LT or in the unavailability tables, as the affected columns (e.g. `Wind Offshore`, `docstatus`) have been removed. The only exceptions are `Solar` and `Wind Onshore` in the DE_LU (12 values) and FR (27 values) `generation_wind_solar_forecast`, which are DST-related and short gaps to be filled during preprocessing.

In [6]:
for code in codes:
    print(f"===== {code} {"=" * 20}")
    for n, d in data[code].items():
        print_missing_values(d, n)

===== PL ====================

--- [load] Missing values ---
No missing values.

--- [generation] Missing values ---
No missing values.

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
No missing values.

--- [generation_units_unavailability] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== DE_LU ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No missing values.

--- [generation_wind_solar_forecast] Missing values ---
Solar           12
Wind Onshore    12

--- [generation_units_unavailability] Missing values ---
No missing values.

--- [generation_forecast] Missing values ---
No missing values.
===== FR ====================

--- [load_forecast] Missing values ---
No missing values.

--- [day_ahead_prices] Missing values ---
No

## 3. Conclusions

The cleaning decisions from the data quality audit have been applied to PL, DE_LU, FR and LT, and the checks above confirm the result. The cleaned datasets of all bidding zones, including those that required no cleaning, are saved to `data/clean/entsoe/` and verified by reading the files back.

## 4. Saving Clean Data

In [7]:
for country, datasets in data.items():
    for name, dataset in datasets.items():
        save_data_entsoe(name, dataset, country, DOWNLOAD_START_DATE, DOWNLOAD_END_DATE)

INFO:src.utils.cleaning:Saved clean/entsoe/AT/2022-12-01_2026-09-16/load_forecast.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/AT/2022-12-01_2026-09-16/water_reservoirs_hydro_storage.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/AT/2022-12-01_2026-09-16/day_ahead_prices.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/AT/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/AT/2022-12-01_2026-09-16/generation_forecast.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/load_forecast.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/day_ahead_prices.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/CZ/2022-12-01_2026-09-16/generation_forecast.parquet
INFO:src.utils.cleaning:Saved clean/entsoe/DE_LU/2022-12-01_2026-09-16/load_forecast.parquet
INFO:src.utils.cleaning:Save

In [8]:
bidding_zones = data.keys()
saved_count = {k: 0 for k in bidding_zones}

for bidding_zone in bidding_zones:
    data_dir = (
        DATA_DIR
        / "clean"
        / "entsoe"
        / bidding_zone
        / f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
    )

    file_paths = list(data_dir.iterdir())
    saved_count[bidding_zone] = len(file_paths)

    print(f"\n--- {bidding_zone} ({len(file_paths)} file/s) " + "-" * 30)

    for fp in file_paths:
        df = pd.read_parquet(fp)
        rel_path = fp.relative_to(BASE_DIR)
        print(
            f"  [FILE] {rel_path} | shape: {df.shape} | idx: {df.index.min()}-{df.index.max()}"
        )


--- AT (5 file/s) ------------------------------
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/load_forecast.parquet | shape: (132956, 1) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/water_reservoirs_hydro_storage.parquet | shape: (196, 1) | idx: 2022-11-27 23:00:00+00:00-2026-08-30 23:00:00+00:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/day_ahead_prices.parquet | shape: (58437, 1) | idx: 2022-12-01 00:00:00+01:00-2026-09-16 00:00:00+02:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/generation_wind_solar_forecast.parquet | shape: (132956, 2) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00
  [FILE] data/clean/entsoe/AT/2022-12-01_2026-09-16/generation_forecast.parquet | shape: (54836, 2) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00

--- CZ (4 file/s) ------------------------------
  [FILE] data/clean/entsoe/CZ/2022-12-01_2026-09-16/load_forecast.parquet | shape: (91409, 1